# nano-prover — fixed Kaggle runner

**One-time setup:** right panel → Accelerator **GPU T4 x2**, Internet **On**; Add-ons → Secrets → add
**`GITHUB_TOKEN`** (GitHub token, scope `repo`).

**Every run:** press ▶ on the cell below. It pulls the latest repo, **trains and pushes the model
right away** (so a 12 h timeout during the slow generation can never lose it), then generates eval
proofs and pushes them. Change the next run by editing `training/scripts/run_kaggle.sh` and `git push`.
When done, locally: `git pull` (model is in `training/model/`) and `python -m assess`.

In [ ]:
# ---- FIXED: clone → train → PUSH MODEL → gen → push preds.  Do not edit (edit run_kaggle.sh). ----
import os
os.chdir('/kaggle/working')
!rm -rf Conjecture-Gym && git clone --depth 1 https://github.com/FELIXGAO0315/Conjecture-Gym.git
os.chdir('/kaggle/working/Conjecture-Gym')
!nvidia-smi -L
from kaggle_secrets import UserSecretsClient
TOKEN = UserSecretsClient().get_secret('GITHUB_TOKEN')
REMOTE = f'https://{TOKEN}@github.com/FELIXGAO0315/Conjecture-Gym.git'

# 1) train, then IMMEDIATELY push the model — a timeout during gen can no longer lose it
!bash training/scripts/run_kaggle.sh train
!git add -f training/model
!git -c user.email=kaggle@run.local -c user.name=kaggle commit -m 'trained adapter (pushed post-train)' || true
!git push {REMOTE} main
print('>>> MODEL SAVED to training/model/ and pushed. The deliverable is safe.')

# 2) generate eval proofs, then push preds (best-effort; the model is already safe above)
!bash training/scripts/run_kaggle.sh gen
!git add -f training/scripts/preds.jsonl
!git -c user.email=kaggle@run.local -c user.name=kaggle commit -m 'preds' || true
!git push {REMOTE} main
print('\nDONE. Locally:  git pull  &&  python -m assess')